# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zainabaon/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/zainabaon/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    os.chdir("/content")
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "CSV not found"
print("Ready.")

Working dir: /content/flyrank-ml-internship
Ready.


My Lane 2 task is a scoring / ranking problem, built on top of a binary classification model. The output isn't a single yes/no decision on one page — it's an ordered queue of all pages, ranked by refresh-priority, so a reviewer with limited capacity works top-down. Underneath the ranking, I use a classifier that estimates the probability a page is declining (or will decline); that probability is the score I rank pages by. This matches what I saw directly in notebooks 01 and 02: the pipeline trains a classifier (random forest) and then sorts pages by predicted probability to build the review queue — classification is the mechanism, but ranking is the actual task shape, since the deliverable is an ordered list, not isolated predictions.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


My proxy target, matching the starter pipeline, is: is_declining_label = (trend_direction == "down"). This is a same-window proxy, not a future outcome — it labels a page as "declining" based on a trend bucket already computed from the current data, not based on what happens next. I'm using it as my starting proxy because it's simple, transparent, and already validated in the starter pipeline (Precision@50 = 0.740 with random forest). A stronger target for later in the capstone would be a genuinely future-looking one: features from a prior window (e.g. days_since_last_update, impressions_90d, ctr, avg_position measured before a cutoff date) predicting whether the page declines over a future window (e.g. the next 30 days) — this avoids the "proxy is really just the same-time label" weakness and gives a real forecasting task.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


My success metric is Precision@K, specifically Precision@50 (matching realistic review capacity of ~50 pages per cycle), with Average Precision as a secondary check on the overall ranking quality. Precision@K is the right metric here — not accuracy — because the deliverable is a ranked queue that a reviewer works through from the top, and what matters is how many of the top K flagged pages are genuinely worth reviewing, not how the model performs across the entire dataset (most of which will never be looked at). The starter pipeline already demonstrates this: hand-written rule Precision@50 = 0.240 vs random forest Precision@50 = 0.740 — a metric like plain accuracy would obscure this difference, since both methods see the same overall class balance.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


One row = one content page, at the current snapshot (90-day trailing window). Below is a real slice of the dataframe showing the columns relevant to my lane, and what the target column looks like.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Build the proxy target
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# The columns relevant to Lane 2 scoring
lane_cols = [
    "content_id", "client_id", "trend_direction", "is_declining_label",
    "impressions_90d", "days_since_last_update", "avg_position",
    "ctr", "word_count", "content_age_days"
]

print("Unit of analysis: one row = one content page")
print("Shape:", df[lane_cols].shape)
df[lane_cols].head(10)


Unit of analysis: one row = one content page
Shape: (30000, 10)


,content_id,client_id,trend_direction,is_declining_label,impressions_90d,days_since_last_update,avg_position,ctr,word_count,content_age_days
0,content_304f48230142,client_f369cb89fc,down,1,3803,20,10.6,0.76,3221.0,187
1,content_a1fb4e703a9e,client_4e07408562,down,1,15320,25,20.3,0.05,2481.0,445
2,content_9aa793d4d895,client_7f2253d7e2,down,1,12581,20,36.5,0.09,3515.0,141
3,content_331d6c4de07b,client_19581e27de,stable,0,11751,22,6.2,0.49,NaN,463
4,content_d99b7a2d90ca,client_3fdba35f04,down,1,19140,14,44.0,0.13,2803.0,263
5,content_d4084a4bc775,client_f369cb89fc,down,1,3970,20,8.5,0.03,3080.0,147
6,content_9a34b442b552,client_8722616204,down,1,20,20,7.0,0.00,3059.0,90
7,content_a63219c6e95a,client_19581e27de,stable,0,1724,22,21.2,0.06,NaN,445
8,content_5e6c160719bc,client_6208ef0f77,down,1,32574,20,46.0,0.09,3807.0,90
9,content_c27558df2b0c,client_19581e27de,down,1,1240,104,4.9,0.16,NaN,257


A fixed rule (like "stale AND visible AND above N impressions") can only combine a handful of thresholds a human picks by hand, and it treats every qualifying page as equally important once it clears those thresholds. This is measurably worse in this data: the hand-written rule scored Precision@50 = 0.240, while a random forest — which can learn nonlinear combinations and relative weightings across many signals (impressions, position, CTR, age, word count, etc.) — scored 0.740, roughly 3x better. That gap shows the real relationships between these signals and decline are more complex than a short hand-written rule can capture; a model can weigh dozens of signals and their interactions simultaneously, and rank by a continuous probability instead of a hard in/out threshold, which is exactly what a "review the most promising 50 first" workflow needs.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.